In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

In [15]:
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

In [16]:
users_df = spark.read.csv("data/user_table.csv", header=True, inferSchema=True)

In [17]:
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
])

kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()

In [18]:
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"), tx_schema).alias("tx")).select("tx.*")
fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

In [19]:
enriched_fraud = fraud_stream.join(users_df, "userId")

In [20]:
output_stream = enriched_fraud \
    .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")

In [23]:
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "fraud-notification") \
    .option("checkpointLocation", "/workspace/checkpoints") \
    .start()
query.awaitTermination()

26/06/19 04:20:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 04:20:54 WARN StreamingQueryManager: Stopping existing streaming query [id=ce25ab60-338e-4a53-b630-867851e08c69, runId=a156ee3a-7f4b-4505-9a43-6dc0c05079c3], as a new run is being started.
26/06/19 04:20:54 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^

KeyboardInterrupt: 